In [1]:
from utils.spark_session import get_spark_session
from pyspark.sql.functions import col, avg, stddev, count, round 
import os

In [2]:
spark = get_spark_session(app_name="03-features")

In [3]:
processed_df = spark.read.parquet(os.path.join('..', 'data', 'processed', 'processed.parquet'))

In [4]:
processed_df.show()

+--------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----------+---------------------+---------------------+------------------+--------------------+----------------------------+--------------------+------------------+
|          account_id|time_since_test_start|amount|max_discount|completed_offer_types|real_amount|target_converted|age|credit_card_limit|gender|registered_on|reg_day|reg_month|reg_year|      index|real_amount_per_limit|n_transactions_so_far| amount_cumulative|n_conversions_so_far|pct_current_vs_total_session|    amount_pct_limit|limit_factor_vs_tx|
+--------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----------+---------------------+---------------------+------------------+--------------------+-----------------

In [5]:
processed_df.select("limit_factor_vs_tx", "amount_pct_limit", "pct_current_vs_total_session",
                    "n_conversions_so_far", "amount_cumulative", "n_transactions_so_far", 
                    "real_amount_per_limit", "target_converted").show(5)

+------------------+--------------------+----------------------------+--------------------+------------------+---------------------+---------------------+----------------+
|limit_factor_vs_tx|    amount_pct_limit|pct_current_vs_total_session|n_conversions_so_far| amount_cumulative|n_transactions_so_far|real_amount_per_limit|target_converted|
+------------------+--------------------+----------------------------+--------------------+------------------+---------------------+---------------------+----------------+
|3249.0974729241875|3.077777777777778E-4|                       22.16|                   0|               0.0|                    0| 3.077777777777778E-4|               0|
| 5305.821665438467|1.884722222222222...|          0.6123646209386282|                   0|             22.16|                    1| 1.884722222222222...|               1|
|  4469.27374301676|           2.2375E-4|         0.45088161209068006|                   1|35.730000000000004|                    2|        

In [6]:
# List of features to analyze
features = [
    "limit_factor_vs_tx",
    "amount_pct_limit",
    "pct_current_vs_total_session",
    "n_conversions_so_far",
    "amount_cumulative",
    "n_transactions_so_far",
    "real_amount_per_limit"
]

# Generate aggregation for each feature
for feature in features:
    print(f"\n=== {feature} grouped by target_converted ===")
    processed_df.groupBy("target_converted").agg(
        round(avg(col(feature)), 4).alias("mean"),
        round(stddev(col(feature)), 4).alias("stddev"),
        count("*").alias("count")
    ).orderBy("target_converted").show()



=== limit_factor_vs_tx grouped by target_converted ===
+----------------+----------+----------+------+
|target_converted|      mean|    stddev| count|
+----------------+----------+----------+------+
|               0|17934.2298|51586.0244|108336|
|               1| 3413.2254|  2035.705| 30617|
+----------------+----------+----------+------+


=== amount_pct_limit grouped by target_converted ===
+----------------+-------+------+------+
|target_converted|   mean|stddev| count|
+----------------+-------+------+------+
|               0| -0.279|1.2853|108336|
|               1|-0.4595|4.1257| 30617|
+----------------+-------+------+------+


=== pct_current_vs_total_session grouped by target_converted ===
+----------------+------+------+------+
|target_converted|  mean|stddev| count|
+----------------+------+------+------+
|               0|1.2674|6.4801|108336|
|               1|5.8907| 21.89| 30617|
+----------------+------+------+------+


=== n_conversions_so_far grouped by target_con

In [7]:
spark.stop()